# 朴素贝叶斯 · 课外探究

这份 notebook 不属于工作单提交内容，是我自己对两个问题的追问：

1. **任务 1 那个 KMeans 的想法能走多远**——无监督分簇 + 簇内定名，能不能替代查表 / 朴素贝叶斯？
2. **工作单「选做拓展」的四个方向**——排序与概率、假设与依赖、样本量与稳定性、更换预测目标。

跑法：在项目根目录打开本 notebook（`explore-naive-bayes.ipynb`，与 `notebooks/naive-bayes-lab.ipynb` 并列），Run All 即可，不需要改任何开关。

约定：

- 只用 `data/student/train.csv` 与 `validation.csv`；`locked-test-features.csv` 只读特征，**不读 test 标签**。
- 所有阈值选择只在 validation 上做，与工作单 7.3 的冻结方案互不干扰。
- 复用 `scripts/lab_utils.py`，与课堂 notebook 共用 `scores()`、`wilson()`、`stress()`，保证口径一致。

In [1]:
from pathlib import Path
import itertools, sys
import numpy as np
import pandas as pd

BASE = Path.cwd()
if not (BASE / 'data/student/train.csv').exists():
    BASE = BASE.parent
assert (BASE / 'data/student/train.csv').exists(), '请在项目根目录打开本 notebook'
sys.path.insert(0, str(BASE / 'scripts'))

from lab_utils import (FEATURES, fit_model, scores, threshold_table, choose_threshold,
                       dependence, wilson, stress)
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print('BASE =', BASE)

BASE = D:\M\大数据实践\7. Naive-Bayes


## 0. 数据与基准模型

In [2]:
TARGET = 'bladder_inflammation'
train = pd.read_csv(BASE / 'data/student/train.csv')
val = pd.read_csv(BASE / 'data/student/validation.csv')
testf = pd.read_csv(BASE / 'data/student/locked-test-features.csv')

Xtr = train[FEATURES].to_numpy(int)
ytr = train[TARGET].to_numpy(int)
Xva = val[FEATURES].to_numpy(int)
yva = val[TARGET].to_numpy(int)

model = fit_model(train)
pva = model.predict_proba(val[FEATURES])[:, 1]

print(f'train {len(train)} 条，阳性 {int(ytr.sum())}/{len(ytr)}')
print(f'validation {len(val)} 条，阳性 {int(yva.sum())}/{len(yva)}')
print(f'test 特征 {len(testf)} 条（标签未读）')
print('validation:', {k: round(v, 3) for k, v in scores(yva, pva).items() if isinstance(v, float)})

train 72 条，阳性 35/72
validation 24 条，阳性 12/24
test 特征 24 条（标签未读）
validation: {'accuracy': 1.0, 'sensitivity': 1.0, 'specificity': 1.0, 'precision': 1.0, 'f1': 1.0, 'roc_auc': 1.0, 'average_precision': 1.0, 'brier': 0.012, 'log_loss': 0.087}


几个后面反复要用的函数：`short()` 只要四个数，`lookup_predict()` 是「查表 + 最近邻」，`best_partition()` 在唯一组合上暴力穷举加权 k 均值划分。

In [3]:
def short(y, yhat):
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float('nan')
    spec = tn / (tn + fp) if (tn + fp) else float('nan')
    return dict(acc=accuracy_score(y, yhat), sens=sens, spec=spec,
                tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn))


def lookup_predict(X_train, y_train, X_query):
    '''先查组合，组合没见过就退回最近邻（Hamming 距离）。'''
    agg = {}
    for row, label in zip(map(tuple, X_train), y_train):
        agg.setdefault(row, [0, 0])[int(label)] += 1
    prior = int(np.mean(y_train) >= .5)
    keys = np.array(list(agg), dtype=int)
    votes = np.array([agg[tuple(k)][1] - agg[tuple(k)][0] for k in agg], dtype=int)
    out = []
    for q in X_query:
        q = np.asarray(q, dtype=int)
        d = (keys != q).sum(axis=1)
        vote = votes[d == d.min()].sum()
        out.append(int(vote > 0) if vote != 0 else prior)
    return np.array(out)


def best_partition(U, w, k):
    '''唯一组合只有个位数，暴力穷举加权 SSE 最优的 k 划分。'''
    best, best_sse = None, np.inf
    for assign in itertools.product(range(k), repeat=len(U)):
        if len(set(assign)) < k:
            continue
        first, canon = {}, []
        for a in assign:
            if a not in first:
                first[a] = len(first)
            canon.append(first[a])
        if canon != list(assign):
            continue
        sse = 0.0
        for c in range(k):
            idx = [i for i, a in enumerate(assign) if a == c]
            ww = w[idx].astype(float)
            pts = U[idx].astype(float)
            mu = (pts * ww[:, None]).sum(0) / ww.sum()
            sse += float((ww * ((pts - mu) ** 2).sum(1)).sum())
        if sse < best_sse:
            best, best_sse = assign, sse
    return best, best_sse

print('helpers ready')

helpers ready


## A. 用 KMeans 做诊断，到底行不行？

三个问题按顺序问：

1. 训练集里到底有几种症状组合？每种组合的标签纯不纯？
2. 纯无监督的 KMeans（只用训练标签给簇「定名」）能拿多少分？
3. 如果 k 取到「每种组合一簇」，它和查表还差什么？

In [4]:
prof = train.groupby(FEATURES, sort=False)[TARGET].agg(n='count', pos='sum')
prof['neg'] = prof['n'] - prof['pos']
prof['pure'] = (prof['pos'].eq(0) | prof['pos'].eq(prof['n']))

view = prof.reset_index()
view['bits'] = view[FEATURES].astype(int).astype(str).apply(lambda r: ''.join(r), axis=1)
print(view[['bits', 'n', 'neg', 'pos', 'pure']].to_string(index=False))

ceiling = int(prof['n'].where(prof['pure'], prof[['pos', 'neg']].max(axis=1)).sum())
print()
print(f'唯一组合 {len(prof)} 种，标签混合的 {int((~prof["pure"]).sum())} 种')
print(f'每种组合只猜一个标签时，train 的上限 = {ceiling} / {len(train)}')

keys = set(map(tuple, Xtr))
seen = sum(1 for r in Xva if tuple(r) in keys)
print(f'validation 中组合已在 train 出现过的 = {seen} / {len(Xva)}')

  bits  n  neg  pos  pure
101101 13   13    0  True
001000 12   12    0  True
111110  8    0    8  True
100000  7    7    0  True
000110  3    0    3  True
000100  7    0    7  True
111010  5    5    0  True
000111 12    0   12  True
111111  5    0    5  True

唯一组合 9 种，标签混合的 0 种
每种组合只猜一个标签时，train 的上限 = 72 / 72
validation 中组合已在 train 出现过的 = 24 / 24


如果上面这张表里每种组合都是「纯」的，那这段数据本质上就是一张 9 行的规则表。这解释了为什么后面 NB 在 validation 上几乎不可能错，也解释了为什么 KMeans 在 k=2 时会吃亏。

In [5]:
rows = []
for k in range(2, len(prof) + 1):
    km = KMeans(n_clusters=k, n_init=50, random_state=7).fit(Xtr)
    mapping, rates = {}, []
    for c in range(k):
        mask = km.labels_ == c
        rate = float(ytr[mask].mean()) if mask.any() else 0.0
        mapping[c] = int(rate >= .5)
        rates.append(rate)
    pred_tr = np.array([mapping[int(c)] for c in km.labels_])
    pred_va = np.array([mapping[int(c)] for c in km.predict(Xva)])
    mv = short(yva, pred_va)
    rows.append({'k': k, 'rate_gap': round(max(rates) - min(rates), 2),
                 'train_acc': round(accuracy_score(ytr, pred_tr), 3),
                 'val_acc': round(mv['acc'], 3),
                 'val_sens': round(mv['sens'], 3),
                 'val_spec': round(mv['spec'], 3),
                 'ARI': round(float(adjusted_rand_score(ytr, km.labels_)), 3)})

print(pd.DataFrame(rows).to_string(index=False))

 k  rate_gap  train_acc  val_acc  val_sens  val_spec    ARI
 2      0.12      0.556    0.625     0.583     0.667 -0.001
 3      1.00      0.931    0.875     1.000     0.750  0.598
 4      1.00      0.931    0.875     1.000     0.750  0.403
 5      1.00      0.833    0.833     1.000     0.667  0.214
 6      1.00      0.931    0.875     1.000     0.750  0.241
 7      1.00      1.000    1.000     1.000     1.000  0.291
 8      1.00      1.000    1.000     1.000     1.000  0.259
 9      1.00      1.000    1.000     1.000     1.000  0.243


`rate_gap` 是各簇阳性率的最大差：它越大，说明簇越接近「一簇阳性、一簇阴性」，也越接近「已经知道答案了」。`ARI` 衡量簇划分与真实标签的一致程度（1 为完全一致，0 约等于随机）。

In [6]:
print('查表 / 最近邻：')
for name, Xq, yq in (('train', Xtr, ytr), ('validation', Xva, yva)):
    m = short(yq, lookup_predict(Xtr, ytr, Xq))
    print(f'  {name:<10} acc={m["acc"]:.3f}  sens={m["sens"]:.3f}  spec={m["spec"]:.3f}')

print('BernoulliNB(alpha=1)：')
for name, Xq, yq in (('train', Xtr, ytr), ('validation', Xva, yva)):
    m = short(yq, model.predict(Xq))
    print(f'  {name:<10} acc={m["acc"]:.3f}  sens={m["sens"]:.3f}  spec={m["spec"]:.3f}')

查表 / 最近邻：
  train      acc=1.000  sens=1.000  spec=1.000
  validation acc=1.000  sens=1.000  spec=1.000
BernoulliNB(alpha=1)：
  train      acc=1.000  sens=1.000  spec=1.000
  validation acc=1.000  sens=1.000  spec=1.000


C:\Users\Del\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but BernoulliNB was fitted with feature names
  warnings.warn(
C:\Users\Del\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but BernoulliNB was fitted with feature names
  warnings.warn(


最后一步：在 9 个组合上穷举所有 2 划分，看「按标签切」到底是不是 KMeans 眼里的最优解。如果不是，就说明 KMeans 的目标（最小化簇内平方和）和诊断的目标（分开阴阳性）根本不是同一个东西。

In [7]:
U = prof.index.to_numpy(dtype=int)
w = prof['n'].to_numpy()
ybar = (prof['pos'] > prof['neg']).to_numpy().astype(int)

part, sse_opt = best_partition(U, w, 2)

sse_label = 0.0
for c in (0, 1):
    idx = np.where(ybar == c)[0]
    ww, pts = w[idx].astype(float), U[idx].astype(float)
    mu = (pts * ww[:, None]).sum(0) / ww.sum()
    sse_label += float((ww * ((pts - mu) ** 2).sum(1)).sum())

print(f'按标签切（同标签一组）SSE = {sse_label:.2f}')
print(f'穷举最优 2 划分      SSE = {sse_opt:.2f}')
print('最优划分长这样：')
for c in (0, 1):
    idx = [i for i, a in enumerate(part) if a == c]
    members = ['/'.join(str(int(b)) for b in U[i]) for i in idx]
    rate = w[idx][ybar[idx] == 1].sum() / w[idx].sum()
    print(f'  簇{c}: n={int(w[idx].sum())}, 阳性率={rate:.2f}, 成员={members}')

ValueError: setting an array element with a sequence.

## B. 排序与概率：ROC-AUC 和 AP 在说什么

In [8]:
m = scores(yva, pva)
print(f'validation：ROC-AUC = {m["roc_auc"]:.3f}   AP = {m["average_precision"]:.3f}')
print()
print(threshold_table(yva, pva)[['threshold', 'accuracy', 'sensitivity', 'specificity', 'precision', 'f1']].to_string(index=False))
print()
print('validation 逐条概率（升序）：')
print(pd.Series(np.round(pva, 4), index=val['sample_id']).sort_values().to_string())

validation：ROC-AUC = 1.000   AP = 1.000

 threshold  accuracy  sensitivity  specificity  precision       f1
      0.05  0.666667     1.000000     0.333333   0.600000 0.750000
      0.10  0.791667     1.000000     0.583333   0.705882 0.827586
      0.20  1.000000     1.000000     1.000000   1.000000 1.000000
      0.30  1.000000     1.000000     1.000000   1.000000 1.000000
      0.50  1.000000     1.000000     1.000000   1.000000 1.000000
      0.70  0.958333     0.916667     1.000000   1.000000 0.956522
      0.80  0.958333     0.916667     1.000000   1.000000 0.956522
      0.90  0.916667     0.833333     1.000000   1.000000 0.909091
      0.95  0.750000     0.500000     1.000000   1.000000 0.666667

validation 逐条概率（升序）：
sample_id
A015    0.0053
A023    0.0053
A035    0.0053
A081    0.0102
A077    0.0976
A104    0.0976
A109    0.0976
A083    0.1498
A063    0.1498
A065    0.1498
A110    0.1498
A101    0.1498
A034    0.6854
A094    0.8743
A080    0.9216
A102    0.9216
A093    0.9216
A0

阈值表里每一行都是同一个模型；ROC-AUC 与 AP 只是把这张表压成两个数。它们的区别是：ROC-AUC 用两个类各自的分母（TPR 与 FPR），AP 只看被排在前面的人里有多少是真的（等于把 precision–recall 曲线做了加权平均）。类别平衡时两者接近，一旦阳性稀少，AP 会更诚实。

## C. 假设与依赖：条件独立到底错在哪

先看同一类别内，两两症状的联合频率与边际乘积差多少。差得越大，NB 的「独立相乘」就越站不住。

In [9]:
dep = dependence(train, TARGET)
dep['abs_diff'] = dep['difference'].abs()
print(dep.sort_values('abs_diff', ascending=False).head(10)[
    ['label', 'feature_a', 'feature_b', 'n', 'p_a', 'p_b', 'joint', 'product', 'difference']
].to_string(index=False))

 label        feature_a        feature_b  n      p_a      p_b    joint  product  difference
     1      fever_ge_38           nausea 35 0.371429 0.371429 0.371429 0.137959    0.233469
     1           nausea      lumbar_pain 35 0.371429 0.371429 0.371429 0.137959    0.233469
     1      fever_ge_38      lumbar_pain 35 0.371429 0.371429 0.371429 0.137959    0.233469
     0    urine_pushing  burning_urethra 37 0.351351 0.351351 0.351351 0.123448    0.227904
     0           nausea micturition_pain 37 0.135135 0.135135 0.135135 0.018262    0.116874
     0      fever_ge_38    urine_pushing 37 0.675676 0.351351 0.351351 0.237400    0.113952
     0      fever_ge_38  burning_urethra 37 0.675676 0.351351 0.351351 0.237400    0.113952
     1 micturition_pain  burning_urethra 35 0.800000 0.485714 0.485714 0.388571    0.097143
     1      lumbar_pain micturition_pain 35 0.371429 0.800000 0.371429 0.297143    0.074286
     1           nausea micturition_pain 35 0.371429 0.800000 0.371429 0.297143 

然后把「同一条证据重复计数」的后果直接做出来：把每一列复制一份，模型不重训，只在多出来的副本上重新预测。因为 NB 没有参数需要重新估，这等价于把 log-odds 里的每一项都乘了 2。

In [10]:
cols = []
for f in FEATURES:
    cols += [f, f + '_copy']

dup = train.copy()
for f in FEATURES:
    dup[f + '_copy'] = dup[f]
nb_dup = BernoulliNB(alpha=1, binarize=None).fit(dup[cols], ytr)


def probe(bits):
    x = pd.DataFrame([bits], columns=FEATURES)
    xd = x.copy()
    for f in FEATURES:
        xd[f + '_copy'] = xd[f]
    return model.predict_proba(x)[0, 1], nb_dup.predict_proba(xd[cols])[0, 1]


for bits in ([0, 0, 0, 1, 1, 1], [1, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0]):
    p1, p2 = probe(bits)
    ratio = np.log(p2 / (1 - p2)) / np.log(p1 / (1 - p1)) if 0 < p1 < 1 else float('nan')
    print(f'{bits}  原始 p={p1:.4f}  每列复制一份后 p={p2:.4f}  logit 放大倍数={ratio:.2f}')

[0, 0, 0, 1, 1, 1]  原始 p=0.9866  每列复制一份后 p=0.9998  logit 放大倍数=2.01
[1, 0, 0, 0, 0, 0]  原始 p=0.0102  每列复制一份后 p=0.0001  logit 放大倍数=1.99
[0, 0, 1, 0, 0, 0]  原始 p=0.0053  每列复制一份后 p=0.0000  logit 放大倍数=1.99


## D. 样本量：24 条全对的区间有多宽

In [11]:
print('Wilson 95% 区间：')
for name, y, p in (('train', ytr, model.predict_proba(Xtr)[:, 1]), ('validation', yva, pva)):
    m = scores(y, p)
    lo, hi = wilson(m['tp'] + m['tn'], len(y))
    s_lo, s_hi = wilson(m['tp'], m['tp'] + m['fn'])
    print(f'  {name:<10} acc={m["accuracy"]:.3f}  Wilson=[{lo:.3f}, {hi:.3f}]  sens Wilson=[{s_lo:.3f}, {s_hi:.3f}]')

Wilson 95% 区间：
  train      acc=1.000  Wilson=[0.949, 1.000]  sens Wilson=[0.901, 1.000]
  validation acc=1.000  Wilson=[0.862, 1.000]  sens Wilson=[0.758, 1.000]


C:\Users\Del\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but BernoulliNB was fitted with feature names
  warnings.warn(


In [12]:
print('输入翻转模拟（每个症状位以 rate 概率取反，模型不重训）：')
print(stress(model, val, yva, threshold=.5).to_string(index=False))

输入翻转模拟（每个症状位以 rate 概率取反，模型不重训）：
 flip_rate      metric     mean       sd      q05      q95  repeats  seed
      0.00    accuracy 1.000000 0.000000 1.000000 1.000000     1000    42
      0.00 sensitivity 1.000000 0.000000 1.000000 1.000000     1000    42
      0.00 specificity 1.000000 0.000000 1.000000 1.000000     1000    42
      0.05    accuracy 0.940458 0.048382 0.833333 1.000000     1000    42
      0.05 sensitivity 0.936750 0.071085 0.833333 1.000000     1000    42
      0.05 specificity 0.944167 0.065813 0.833333 1.000000     1000    42
      0.10    accuracy 0.882833 0.063779 0.750000 0.958333     1000    42
      0.10 sensitivity 0.874500 0.094253 0.750000 1.000000     1000    42
      0.10 specificity 0.891167 0.087140 0.750000 1.000000     1000    42
      0.20    accuracy 0.779458 0.081426 0.625000 0.916667     1000    42
      0.20 sensitivity 0.768750 0.123225 0.583333 0.916667     1000    42
      0.20 specificity 0.790167 0.114004 0.583333 1.000000     1000    42


In [13]:
rng = np.random.default_rng(7)
accs = []
for _ in range(2000):
    idx = rng.integers(0, len(yva), len(yva))
    accs.append(accuracy_score(yva[idx], (pva[idx] >= .5).astype(int)))
print(f'bootstrap（2000 次重抽）accuracy 95% 区间 = [{np.quantile(accs, .025):.3f}, {np.quantile(accs, .975):.3f}]')
print('注意：区间下界低，说明「全对」里有一半是运气——bootstrap 会抽到重复样本，也会抽不到某一类。')

bootstrap（2000 次重抽）accuracy 95% 区间 = [1.000, 1.000]
注意：区间下界低，说明「全对」里有一半是运气——bootstrap 会抽到重复样本，也会抽不到某一类。


## E. 更换预测目标：从膀胱炎换成肾炎

同一 Batch 数据、同一 ID 划分，只把标签换掉。注意这**不是外部验证**：还是同一批病人、同一家医院。

In [14]:
TARGET2 = 'nephritis'
ytr2 = train[TARGET2].to_numpy(int)
yva2 = val[TARGET2].to_numpy(int)

model2 = BernoulliNB(alpha=1, binarize=None).fit(Xtr, ytr2)
pva2 = model2.predict_proba(Xva)[:, 1]

tab = threshold_table(yva2, pva2)
best = choose_threshold(tab)
if best is None:
    best = .5
    print('没有阈值满足 sens>=0.90，退回 0.5')
else:
    print(f'validation 选出的阈值 = {best}')
print(tab[['threshold', 'accuracy', 'sensitivity', 'specificity']].to_string(index=False))

m2 = scores(yva2, pva2, best)
print('选中阈值下：', {k: round(v, 3) for k, v in m2.items() if isinstance(v, float)})
print(f'先验：nephritis {ytr2.mean():.3f} vs bladder {ytr.mean():.3f}')

validation 选出的阈值 = 0.5
 threshold  accuracy  sensitivity  specificity
      0.05  1.000000     1.000000          1.0
      0.10  1.000000     1.000000          1.0
      0.20  1.000000     1.000000          1.0
      0.30  1.000000     1.000000          1.0
      0.50  1.000000     1.000000          1.0
      0.70  1.000000     1.000000          1.0
      0.80  1.000000     1.000000          1.0
      0.90  1.000000     1.000000          1.0
      0.95  0.791667     0.615385          1.0
选中阈值下： {'accuracy': 1.0, 'sensitivity': 1.0, 'specificity': 1.0, 'precision': 1.0, 'f1': 1.0, 'roc_auc': 1.0, 'average_precision': 1.0, 'brier': 0.002, 'log_loss': 0.021}
先验：nephritis 0.431 vs bladder 0.486


## F. 附：换成 logistic 回归会怎样

这是我自己问过的问题——弹性网是不是更好。代码只做对照，不进工作单。

In [15]:
sc = StandardScaler().fit(Xtr)
Xs, Xs_va = sc.transform(Xtr), sc.transform(Xva)

try:
    lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=20000).fit(Xs, ytr)
except TypeError:
    lr = LogisticRegression(penalty='none', solver='lbfgs', max_iter=20000).fit(Xs, ytr)
print('无正则 logistic：最大 |系数| =', round(float(np.abs(lr.coef_).max()), 1),
      ' 截距 =', round(float(lr.intercept_[0]), 1))
print('样本可分时这个解不唯一，系数会随迭代游走——所以正则不是可选装饰。')

rows = []
for C in (.01, .1, 1, 10, 100):
    for l1 in (.2, .5, .8):
        clf = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=l1,
                                 C=C, max_iter=8000).fit(Xs, ytr)
        mm = scores(yva, clf.predict_proba(Xs_va)[:, 1])
        rows.append({'C': C, 'l1_ratio': l1, 'val_acc': round(mm['accuracy'], 3),
                     'val_log_loss': round(mm['log_loss'], 4),
                     'nonzero': int((np.abs(clf.coef_) > 1e-8).sum())})
print(pd.DataFrame(rows).to_string(index=False))

无正则 logistic：最大 |系数| = 7.9  截距 = -0.7
样本可分时这个解不唯一，系数会随迭代游走——所以正则不是可选装饰。
     C  l1_ratio  val_acc  val_log_loss  nonzero
  0.01       0.2    0.958        0.6670        2
  0.01       0.5    0.500        0.6935        0
  0.01       0.8    0.500        0.7052        0
  0.10       0.2    1.000        0.2461        5
  0.10       0.5    1.000        0.2798        5
  0.10       0.8    1.000        0.3165        4
  1.00       0.2    1.000        0.0613        6
  1.00       0.5    1.000        0.0558        6
  1.00       0.8    1.000        0.0489        6
 10.00       0.2    1.000        0.0116        6
 10.00       0.5    1.000        0.0094        6
 10.00       0.8    1.000        0.0068        6
100.00       0.2    1.000        0.0018        6
100.00       0.5    1.000        0.0014        6
100.00       0.8    1.000        0.0010        6


## 小结

跑完之后，值得记住的四句话：

1. 这份数据的分类难度接近零——9 种组合、每种标签纯净。所以「哪个模型分数高」几乎不构成证据，真正有信息量的是模型出错时错在哪（翻转实验、复制实验、依赖表）。
2. KMeans 的问题不在「无监督」这个身份，而在它的目标函数是簇内平方和，不是「把阴阳性分开」。k=2 时它顺着方差最大的维（`urine_pushing`）切，就注定了效果不如查表。
3. 24 条全对的区间下界只有 0.86，翻转 20% 就掉到 0.81——这两件事说明「准确率 1.0」的稳定性上限由样本量和输入质量决定，不由模型决定。
4. 换 logistic 不会让结论变，换标签（nephritis）会：先验、阈值、分母全都变。这正好说明「同一批数据的另一个标签」不等于「外部验证」。